# Ablation plots

In [ ]:
import math
import re
import tempfile
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PAPER_STYLE = {
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.04,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.labelsize": 10,
    "axes.titlesize": 11,
    "axes.titleweight": "normal",
    "legend.fontsize": 9,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "lines.linewidth": 1.6,
    "patch.linewidth": 0.4,
}

DATASET_LABELS = {
    "doris": "DORIS",
    "itm": "ITM-Rec",
    "itm-rec": "ITM-Rec",
    "mars": "MARS",
}
VAR2COMP = {
    "no_sequence": "Secuencia",
    "no_graph": "Grafo",
    "no_features": "Atributos",
    "no_context": "Contexto",
    "no_routers": "Fusión",
    "no_gcl": "Contraste",
    "dot_product": "Scoring",
}
COMP_ORDER = [
    "Secuencia",
    "Grafo",
    "Atributos",
    "Contexto",
    "Fusión",
    "Contraste",
    "Scoring",
]
EXCLUDE_VARIANTS = {"full", "base"}

In [ ]:
def parse_mean_std(value):
    if pd.isna(value):
        return np.nan, np.nan
    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value), np.nan
    parts = re.split(r"\s*(?:±|\+/-)\s*", str(value).strip())
    return float(parts[0]) if parts else np.nan, float(parts[1]) if len(
        parts
    ) > 1 else np.nan


def discover_csv_files(input_path):
    p = input_path.expanduser().resolve()
    if p.is_file() and p.suffix.lower() == ".csv":
        return [p]
    if p.is_dir():
        return sorted(p.glob("*.csv"))
    if p.is_file() and p.suffix.lower() == ".zip":
        tmp = Path(tempfile.mkdtemp(prefix="abl_csvs_"))
        with zipfile.ZipFile(p) as zf:
            zf.extractall(tmp)
        return sorted(tmp.rglob("*.csv"))
    raise FileNotFoundError(f"No encuentro CSVs en: {input_path}")


def infer_dataset_name(csv_path):
    stem = csv_path.stem.lower()
    for key, label in DATASET_LABELS.items():
        if key in stem:
            return label
    return csv_path.stem.replace("_", " ").title()


def component_name(variant):
    variant = str(variant)
    return VAR2COMP.get(variant) or variant.replace("_", " ").title()


def compute_importance(
    csv_path,
    metric,
    variant_col="variant",
    seed_col="seed",
    full_variant="full",
    exclude_variants=EXCLUDE_VARIANTS,
    clip_negative=False,
):
    df = pd.read_csv(csv_path)
    for col in [variant_col, metric]:
        if col not in df.columns:
            raise ValueError(f'{csv_path.name}: no existe la columna "{col}".')
    if full_variant not in set(df[variant_col].astype(str)):
        raise ValueError(f'{csv_path.name}: no encuentro full="{full_variant}".')

    ds = infer_dataset_name(csv_path)
    exclude = set(exclude_variants)
    rows = []

    if seed_col in df.columns and pd.api.types.is_numeric_dtype(df[metric]):
        full = (
            df[df[variant_col].astype(str) == full_variant]
            .set_index(seed_col)[metric]
            .astype(float)
        )
        for variant, grp in df.groupby(variant_col):
            variant = str(variant)
            if variant in exclude:
                continue
            abl = grp.set_index(seed_col)[metric].astype(float)
            common = full.index.intersection(abl.index)
            if len(common) == 0:
                continue
            d = full.loc[common] - abl.loc[common]
            if clip_negative:
                d = d.clip(lower=0)
            rows.append(
                {
                    "dataset": ds,
                    "variant": variant,
                    "component": component_name(variant),
                    "importance_mean": float(d.mean()),
                    "importance_std": float(d.std(ddof=1)) if len(d) > 1 else 0.0,
                    "n_seeds": len(d),
                    "metric": metric,
                }
            )
    else:
        stats = {
            str(r[variant_col]): parse_mean_std(r[metric]) for _, r in df.iterrows()
        }
        fm, fs = stats[full_variant]
        for variant, (m, s) in stats.items():
            if variant in exclude:
                continue
            dm = fm - m
            if clip_negative:
                dm = max(0.0, dm)
            ds_ = (
                math.sqrt(fs**2 + s**2)
                if np.isfinite(fs) and np.isfinite(s)
                else np.nan
            )
            rows.append(
                {
                    "dataset": ds,
                    "variant": variant,
                    "component": component_name(variant),
                    "importance_mean": float(dm),
                    "importance_std": float(ds_) if np.isfinite(ds_) else np.nan,
                    "n_seeds": np.nan,
                    "metric": metric,
                }
            )
    return pd.DataFrame(rows)


def collect_importance(input_path, metric, clip_negative=False, exclude_components=()):
    csv_files = discover_csv_files(input_path)
    result = pd.concat(
        [
            compute_importance(p, metric=metric, clip_negative=clip_negative)
            for p in csv_files
        ],
        ignore_index=True,
    )
    result["component"] = pd.Categorical(
        result["component"], categories=COMP_ORDER, ordered=True
    )
    if exclude_components:
        excluded = {
            str(c).strip().casefold() for c in exclude_components if str(c).strip()
        }
        if excluded:
            result = result[
                ~result["component"]
                .astype(str)
                .str.strip()
                .str.casefold()
                .isin(excluded)
            ]
    return result.sort_values(["dataset", "component", "variant"])


def savefig(outdir, name, formats, dpi=300):
    outdir.mkdir(parents=True, exist_ok=True)
    for fmt in formats:
        plt.savefig(outdir / f"{name}.{fmt}", bbox_inches="tight", dpi=dpi)


def apply_plot_style():
    sns.set_theme(
        context="paper",
        style="whitegrid",
        palette="bright",
        font="DejaVu Sans",
        rc=PAPER_STYLE,
    )


def style_axis(ax, grid_axis):
    ax.set_axisbelow(True)
    ax.grid(axis=grid_axis, color="0.88", linewidth=0.7)
    ax.grid(axis="x" if grid_axis == "y" else "y", visible=False)
    sns.despine(ax=ax, trim=True)
    ax.tick_params(length=3, width=0.7, pad=5)


def set_lims(ax, values, axis):
    f = np.asarray([v for v in values if np.isfinite(v)], dtype=float)
    if f.size == 0:
        return
    spread = max(float(f.max()) - float(f.min()), 0.02)
    pad = spread * 0.16
    getattr(ax, f"set_{axis}lim")(
        min(0.0, float(f.min()) - pad), max(0.0, float(f.max()) + pad)
    )


def plot_grouped_bar(data, outdir, metric, formats, show_error_bars=False):
    pivot = (
        data.pivot_table(
            index="component",
            columns="dataset",
            values="importance_mean",
            aggfunc="first",
            observed=False,
        )
        .reindex(COMP_ORDER)
        .dropna(how="all")
    )
    err = data.pivot_table(
        index="component",
        columns="dataset",
        values="importance_std",
        aggfunc="first",
        observed=False,
    ).reindex(pivot.index)
    apply_plot_style()
    fig, ax = plt.subplots(
        figsize=(max(7.2, 0.85 * len(pivot.index) + 2.2), 3.8), layout="constrained"
    )
    ds_list = list(pivot.columns)
    plot_data = data[data["component"].isin(pivot.index)]
    sns.barplot(
        data=plot_data,
        x="component",
        y="importance_mean",
        hue="dataset",
        order=list(pivot.index),
        hue_order=ds_list,
        errorbar=None,
        edgecolor="0.25",
        linewidth=0.45,
        ax=ax,
    )
    if show_error_bars:
        for ctr, ds in zip(ax.containers, ds_list):
            for bar, comp in zip(ctr, pivot.index):
                v, s = pivot.loc[comp, ds], err.loc[comp, ds]
                if pd.notna(v) and pd.notna(s):
                    ax.errorbar(
                        bar.get_x() + bar.get_width() / 2,
                        v,
                        yerr=s,
                        color="0.15",
                        capsize=2.5,
                        elinewidth=0.8,
                        capthick=0.8,
                        fmt="none",
                    )
    ax.axhline(0, color="0.25", linewidth=0.8)
    ax.set_ylabel(f"Caída {metric.upper()} frente al modelo completo")
    ax.set_title(f"Importancia por componente ({metric.upper()})")
    set_lims(ax, plot_data["importance_mean"].values, "y")
    style_axis(ax, "y")
    ax.legend(title=None, frameon=False, ncol=min(len(ds_list), 3), loc="upper right")
    savefig(outdir, f"importance_grouped_bar_{metric.replace('@', '')}", formats)
    plt.close()


def plot_line(data, outdir, metric, formats):
    pivot = (
        data.pivot_table(
            index="component",
            columns="dataset",
            values="importance_mean",
            aggfunc="first",
            observed=False,
        )
        .reindex(COMP_ORDER)
        .dropna(how="all")
    )
    apply_plot_style()
    fig, ax = plt.subplots(
        figsize=(max(7.2, 0.85 * len(pivot.index) + 2.2), 3.8), layout="constrained"
    )
    plot_data = data[data["component"].isin(pivot.index)]
    sns.lineplot(
        data=plot_data,
        x="component",
        y="importance_mean",
        hue="dataset",
        style="dataset",
        markers=True,
        dashes=False,
        hue_order=list(pivot.columns),
        sort=False,
        errorbar=None,
        ax=ax,
    )
    ax.axhline(0, color="0.25", linewidth=0.8)
    ax.set_ylabel(f"Caída {metric.upper()} frente al modelo completo")
    ax.set_title(f"Perfil de ablacion ({metric.upper()})")
    set_lims(ax, plot_data["importance_mean"].values, "y")
    style_axis(ax, "y")
    ax.legend(
        title=None, frameon=False, ncol=min(len(pivot.columns), 3), loc="upper right"
    )
    savefig(outdir, f"importance_line_{metric.replace('@', '')}", formats)
    plt.close()


def plot_individual_barh(
    data, outdir, metric, formats, show_error_bars=False, sort_by="impact"
):
    for ds, subset in data.groupby("dataset", sort=True):
        subset = subset.sort_values(
            "importance_mean" if sort_by == "impact" else "component",
            ascending=sort_by == "impact",
        )
        apply_plot_style()
        fig, ax = plt.subplots(
            figsize=(5.8, max(3.2, 0.42 * len(subset) + 1.1)), layout="constrained"
        )
        labels = subset["component"].astype(str).tolist()
        idx = subset.set_index("component")
        sns.barplot(
            data=subset,
            x="importance_mean",
            y="component",
            order=labels,
            errorbar=None,
            color=sns.color_palette("colorblind")[0],
            edgecolor="0.25",
            linewidth=0.45,
            ax=ax,
        )
        if show_error_bars:
            for bar, comp in zip(ax.containers[0], labels):
                s = idx.loc[comp, "importance_std"]
                if pd.notna(s):
                    ax.errorbar(
                        idx.loc[comp, "importance_mean"],
                        bar.get_y() + bar.get_height() / 2,
                        xerr=s,
                        color="0.15",
                        capsize=2.5,
                        elinewidth=0.8,
                        capthick=0.8,
                        fmt="none",
                    )
        ax.axvline(0, color="0.25", linewidth=0.8)
        ax.set_xlabel(f"Caída {metric.upper()} frente al modelo completo")
        ax.set_title(f"{ds} - importancia por componente")
        set_lims(ax, subset["importance_mean"].values, "x")
        style_axis(ax, "x")
        safe = re.sub(r"[^a-zA-Z0-9]+", "_", ds.lower()).strip("_")
        savefig(
            outdir.parent / safe,
            f"importance_barh_{safe}_{metric.replace('@', '')}",
            formats,
        )
        plt.close()


def export_table(data, outdir, metric):
    outdir.mkdir(parents=True, exist_ok=True)
    t = data.copy()
    t["importance"] = t.apply(
        lambda r: (
            f"{r['importance_mean']:.4f} ± {r['importance_std']:.4f}"
            if pd.notna(r["importance_std"])
            else f"{r['importance_mean']:.4f}"
        ),
        axis=1,
    )
    t.to_csv(outdir / f"importance_values_{metric.replace('@', '')}.csv", index=False)

## Configuration

In [ ]:
def find_project_root(start=Path.cwd()):
    for c in (start.resolve(), *start.resolve().parents):
        if (c / "plots").is_dir() and (c / "notebooks").is_dir():
            return c
    raise FileNotFoundError("Could not locate the EDuRec repository root.")


PROJECT_ROOT = find_project_root()
INPUT_PATH = PROJECT_ROOT / "plots" / "ablation.zip"
PLOTS_DIR = PROJECT_ROOT / "results" / "plots" / "all_datasets"
TABLES_DIR = PROJECT_ROOT / "results" / "tables" / "ablation"
METRIC = "ndcg@20"
FORMATS = ["png", "pdf"]
SHOW_ERROR_BARS = False
CLIP_NEGATIVE = False
SORT_INDIVIDUAL = "impact"
EXCLUDE_COMPONENTS = []

PLOTS_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
print(f"Input:  {INPUT_PATH}")
print(f"Plots:  {PLOTS_DIR.parent / '<dataset>'}")
print(f"Tables: {TABLES_DIR}")

## Load and inspect

In [ ]:
importance = collect_importance(
    INPUT_PATH,
    METRIC,
    clip_negative=CLIP_NEGATIVE,
    exclude_components=EXCLUDE_COMPONENTS,
)
if importance.empty:
    raise ValueError("No components remain after applying the exclusion list.")
importance[
    ["dataset", "component", "variant", "importance_mean", "importance_std", "n_seeds"]
]

## Export table and figures

In [ ]:
export_table(importance, TABLES_DIR, METRIC)
plot_grouped_bar(
    importance, PLOTS_DIR, METRIC, FORMATS, show_error_bars=SHOW_ERROR_BARS
)
plot_line(importance, PLOTS_DIR, METRIC, FORMATS)
plot_individual_barh(
    importance,
    PLOTS_DIR,
    METRIC,
    FORMATS,
    show_error_bars=SHOW_ERROR_BARS,
    sort_by=SORT_INDIVIDUAL,
)

files = sorted(
    p.relative_to(PLOTS_DIR.parent) for p in PLOTS_DIR.parent.rglob("*") if p.is_file()
)
print(f"Generated {len(files)} plot files below {PLOTS_DIR.parent}")
files